# CNN Image Classifier using CIFAR-10

This notebook builds a Convolutional Neural Network (CNN) to classify images into 10 categories using the CIFAR-10 dataset.

**CIFAR-10 Classes:** airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

**Improvements over base model:**
- Batch Normalization for faster, more stable training
- Dropout layers to reduce overfitting
- Adam optimizer from the start (no need for two training phases)
- Prediction pipeline for custom images and Webots camera captures

## 1. Import Libraries and Load Dataset

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
import os

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# Load CIFAR-10 dataset
# 50,000 training images and 10,000 test images, each 32x32 pixels with 3 colour channels (RGB)
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values from 0-255 to 0-1 range
# This helps the model train faster and more effectively
train_images = train_images / 255.0
test_images = test_images / 255.0

# Define class names for CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training images shape: {train_images.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Number of classes: {len(class_names)}")

## 2. Visualize Sample Training Images

In [ ]:
# Display a 5x5 grid of sample training images
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.suptitle("Sample CIFAR-10 Training Images", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Build the CNN Model

Our model has three convolutional blocks, each with:
- **Conv2D layer** — extracts features using learnable filters
- **BatchNormalization** — normalizes layer outputs for faster, more stable training
- **MaxPooling2D** — reduces spatial dimensions, keeping important features
- **Dropout** — randomly disables neurons during training to prevent overfitting

After the convolutional blocks, a Flatten layer converts the 2D feature maps into a 1D vector, followed by Dense (fully connected) layers for classification.

In [ ]:
model = models.Sequential([
    # --- Block 1: 32 filters ---
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Block 2: 64 filters ---
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Block 3: 128 filters ---
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Classification head ---
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10)  # 10 output classes (logits)
])

# Display model architecture and layer details
model.summary()

## 4. Compile and Train the Model

- **Optimizer:** Adam — adaptive learning rate, generally faster than SGD
- **Loss:** SparseCategoricalCrossentropy — suitable for integer labels with multiple classes
- **Epochs:** 15 — enough to reach good accuracy without taking too long (~5-10 minutes on Colab GPU)

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    train_images, train_labels,
    epochs=15,
    validation_data=(test_images, test_labels),
    batch_size=64
)

## 5. Evaluate Model Performance

In [ ]:
# Plot training and validation accuracy
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.3, 1.0])
plt.title('Model Accuracy')
plt.legend(loc='lower right')

# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")
print(f"Test Loss: {test_loss:.4f}")

## 6. Save the Trained Model

In [ ]:
# Save model in Keras format
model.save('cifar10_model.keras')
print("Model saved as 'cifar10_model.keras'")

## 7. Predict on Custom Images

This section loads external images (your own photos or images from the internet) and uses the trained model to predict what object is in the image.

### Instructions:
1. Upload 3 images of your choice (from CIFAR-10 classes) to the same folder as this notebook, or to Colab's file browser
2. Upload 1 image captured from the Webots robot camera
3. Update the filenames in the code below

**Tip:** For best results, use clear images of single objects on a simple background. Good classes to test: airplane, automobile, ship, truck (these tend to classify well).

In [ ]:
def load_and_predict(filename, model, class_names):
    """
    Load an image file, preprocess it to match CIFAR-10 format,
    and return the model's prediction.

    Steps:
    1. Resize image to 32x32 pixels (CIFAR-10 input size)
    2. Convert to numpy array
    3. Normalize pixel values to 0-1
    4. Add batch dimension (model expects batch of images)
    5. Run prediction and return class with highest confidence
    """
    # Load and resize the image to 32x32
    img = image.load_img(filename, target_size=(32, 32))

    # Convert to numpy array (shape: 32, 32, 3)
    img_array = image.img_to_array(img)

    # Normalize pixel values to 0-1 (same as training data)
    img_array = img_array / 255.0

    # Add batch dimension (shape: 1, 32, 32, 3)
    img_batch = np.expand_dims(img_array, axis=0)

    # Get prediction (returns logits for each class)
    predictions = model.predict(img_batch, verbose=0)

    # Convert logits to probabilities using softmax
    probabilities = tf.nn.softmax(predictions[0]).numpy()

    # Get the predicted class index
    predicted_index = np.argmax(probabilities)
    predicted_class = class_names[predicted_index]
    confidence = probabilities[predicted_index] * 100

    return img, predicted_class, confidence, probabilities

print("Prediction function ready.")

In [ ]:
# ============================================================
# UPDATE THESE FILENAMES with your actual image paths
# ============================================================
#
# For Google Colab: upload images using the file browser on the
# left sidebar, then use the filename directly.
#
# For local Jupyter: place images in the same folder as this
# notebook and use the filename, or provide the full file path.
# ============================================================

# 3 images of your own choosing (from CIFAR-10 classes)
my_images = [
    'image1.png',    # <-- Replace with your 1st image filename
    'image2.png',    # <-- Replace with your 2nd image filename
    'image3.png',    # <-- Replace with your 3rd image filename
]

# 1 image captured from the Webots robot camera
webots_image = 'webots_capture.png'  # <-- Replace with your Webots image filename

In [ ]:
# --- Predict on your 3 own images ---

# Load the saved model (so this cell works even if you restart the notebook)
loaded_model = models.load_model('cifar10_model.keras')

plt.figure(figsize=(15, 4))

for i, img_path in enumerate(my_images):
    if not os.path.exists(img_path):
        print(f"WARNING: '{img_path}' not found. Please upload the image.")
        continue

    img, predicted_class, confidence, probs = load_and_predict(img_path, loaded_model, class_names)

    plt.subplot(1, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.1f}%", fontsize=12)
    plt.axis('off')

    print(f"Image {i+1}: {img_path}")
    print(f"  Predicted: {predicted_class} ({confidence:.1f}%)")
    # Show top 3 predictions
    top3 = np.argsort(probs)[-3:][::-1]
    for idx in top3:
        print(f"    {class_names[idx]}: {probs[idx]*100:.1f}%")
    print()

plt.suptitle("Predictions on Own Images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Predict on the Webots camera image ---

if os.path.exists(webots_image):
    img, predicted_class, confidence, probs = load_and_predict(webots_image, loaded_model, class_names)

    plt.figure(figsize=(8, 4))

    # Show the image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Webots Capture\nPrediction: {predicted_class} ({confidence:.1f}%)", fontsize=12)
    plt.axis('off')

    # Show prediction bar chart
    plt.subplot(1, 2, 2)
    colors = ['#2ecc71' if i == np.argmax(probs) else '#95a5a6' for i in range(10)]
    plt.barh(class_names, probs * 100, color=colors)
    plt.xlabel('Confidence (%)')
    plt.title('Prediction Probabilities')
    plt.xlim([0, 100])

    plt.tight_layout()
    plt.show()

    print(f"\nWebots Image: {webots_image}")
    print(f"Predicted: {predicted_class} ({confidence:.1f}%)")
    top3 = np.argsort(probs)[-3:][::-1]
    for idx in top3:
        print(f"  {class_names[idx]}: {probs[idx]*100:.1f}%")
else:
    print(f"WARNING: '{webots_image}' not found. Please upload the Webots capture image.")

## 8. Summary

| Aspect | Details |
|--------|--------|
| **Dataset** | CIFAR-10 (50,000 train / 10,000 test) |
| **Architecture** | 3 Conv blocks (32→64→128 filters) + Dense(128) + Dense(10) |
| **Regularization** | BatchNormalization + Dropout (0.25 / 0.5) |
| **Optimizer** | Adam |
| **Training** | 15 epochs, batch size 64 |
| **Expected Accuracy** | ~75-80% on test set |